# Astro-QuFeX Playground

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
from datetime import datetime

from qmla.config import ConfigError, load_config
from qmla.engine import Trainer

ROOT = Path(".").resolve().parent
CONFIG = "experiments.toml"

RUN_ID = f"notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

ROOT

WindowsPath('G:/MALTA/code/qmla')

In [18]:
config = load_config(ROOT / "configs" / CONFIG)
config.paths.runs_dir

WindowsPath('G:/MALTA/code/qmla/runs')

In [19]:
mode = config.run.model
run_dir = config.paths.runs_dir / RUN_ID

print("Configuration loaded successfully.")
print(f"Mode: {mode}")
print(f"Run directory: {run_dir}")


Configuration loaded successfully.
Mode: qufex
Run directory: G:\MALTA\code\qmla\runs\notebook_20260910_121620


In [20]:
trainer = Trainer(config, run_dir, mode=mode)

RuntimeError: Dataset validation failed: [Errno 2] No such file or directory: 'G:\\MALTA\\code\\qmla\\data\\processed\\gz2-64-697bcf06165e41c4\\dataset_metadata.json'. Run uv run --no-sync python -m scripts.preprocess_data --config configs/experiments.toml --profile full64. Cache: G:\MALTA\code\qmla\data\processed\gz2-64-697bcf06165e41c4

In [32]:
for batch in trainer.train_loader:
    inputs, targets = batch
    inputs = inputs.to(trainer.device, non_blocking=True)
    if trainer.device.type == "cuda":
        inputs = inputs.contiguous(memory_format=trainer.torch.channels_last)
    targets = targets.to(trainer.device, non_blocking=True)
    break

inputs.shape

torch.Size([2, 3, 128, 128])

In [33]:
from torchview import draw_graph
from torchinfo import summary

summary(trainer.model, input_size=(1, 3, 128, 128))

Layer (type:depth-idx)                   Output Shape              Param #
GalaxyClassifier                         [1, 3]                    --
├─Sequential: 1-1                        [1, 8, 8, 8]              --
│    └─ConvBlock: 2-1                    [1, 8, 64, 64]            --
│    │    └─Sequential: 3-1              [1, 8, 64, 64]            232
│    └─ConvBlock: 2-2                    [1, 8, 32, 32]            --
│    │    └─Sequential: 3-2              [1, 8, 32, 32]            592
│    └─ConvBlock: 2-3                    [1, 8, 16, 16]            --
│    │    └─Sequential: 3-3              [1, 8, 16, 16]            592
│    └─ConvBlock: 2-4                    [1, 8, 8, 8]              --
│    │    └─Sequential: 3-4              [1, 8, 8, 8]              592
├─Sequential: 1-2                        [1, 16, 2, 2]             --
│    └─Conv2d: 2-5                       [1, 16, 8, 8]             128
│    └─BatchNorm2d: 2-6                  [1, 16, 8, 8]             32
│    └─ReL

In [34]:
graph = draw_graph(
    trainer.model,
    input_size=(1, 3, 128, 128)
)

graph.visual_graph

RuntimeError: Failed to run torchgraph see error message